# 01 - Create And Expand A Domain-Specific Red-Team Dataset

This notebook is the main dataset-generation utility. It lets you create banking/finance attack prompts today, and reuse the same flow for another domain by swapping the domain pack and config.

The notebook can use all available generation features:

| Feature | What it does | API key needed? | Default |
|---|---|---:|---:|
| Domain pack + taxonomy | Defines the domain, risks, personas, contexts, and framework mappings | No | On |
| Baseline seed prompts | Creates safe starting prompts for each risk category | No | On |
| Seed-source ingestion | Adds starter ideas from OWASP/MITRE/Garak-style source adapters | No | On |
| Garak corpus seeds | Mines installed Garak probe/corpus strings, reduces them, and adapts them to the domain | No | On if enabled in YAML |
| Local mutation orchestrator | Creates deterministic variants such as authority pretext, policy exception, indirect prompt injection, and multi-turn plans | No | On |
| LLM generator | Uses a small LLM to create stronger domain-specific variants from existing records | Yes | Off |
| DeepTeam expansion | Uses DeepTeam local templates or live simulator-backed generation | Local mode: No. LLM mode: Yes | Off |
| Garak scanner expansion | Adds scanner-style vulnerability coverage patterns, optionally from a live Garak target scan | Static mode: No. Live scan: maybe | Off |

Recommended beginner path: run each cell in order with the defaults, review the previews, then turn on one optional live feature at a time.


## 1. Setup

This cell finds the project root, loads `.env`, and imports the local package. It does not call any external model APIs.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path

import yaml
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "src" / "finance_redteam").exists())

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")
print(f"OpenAI key present: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"Google key present: {bool(os.getenv('GOOGLE_API_KEY'))}")


## 2. Optional: Create A New Domain Pack Template

Use this only when you want a new domain, such as healthcare, insurance, HR, legal, or retail.

A domain pack is the reusable domain layer. It contains domain risks, personas, contexts, framework mappings, and seed guidance. After creating a template, edit the generated YAML files and then point `CONFIG_PATH` to the generated config.


In [ ]:
from finance_redteam.domain_pack_template import create_domain_pack_template

CREATE_NEW_DOMAIN_PACK = False

NEW_DOMAIN = {
    "domain_id": "insurance",
    "display_name": "Insurance",
    "description": "Insurance-domain LLM safety benchmark for claims, underwriting, policy servicing, fraud, and customer support.",
    "personas": ["claims adjuster", "underwriter", "customer support agent", "fraud analyst"],
    "contexts": ["claims review", "policy quote", "underwriting", "customer support", "fraud investigation"],
    "risk_count": 8,
}

if CREATE_NEW_DOMAIN_PACK:
    result = create_domain_pack_template(**NEW_DOMAIN, overwrite=False)
    print(json.dumps({
        "domain_pack": str(result.domain_pack_path),
        "config": str(result.config_path),
        "taxonomy": str(result.taxonomy_path),
        "seeds": str(result.seeds_path),
    }, indent=2))
else:
    print("Skipping new domain-pack creation. Using existing config/domain pack.")


## 3. Choose Config And Feature Toggles

This is the main production-control cell.

The defaults below are a **balanced production demonstration profile**. They enable every major dataset-generation capability while keeping hard limits small enough for notebook use:

- baseline domain taxonomy and seed prompts are enabled;
- OWASP/MITRE/Garak-style seed-source ingestion is enabled;
- Garak built-in corpus mining is enabled;
- deterministic local mutation orchestration and multi-turn planning are enabled;
- LLM-based variant generation is enabled with a small OpenAI model;
- DeepTeam live simulator-backed expansion is enabled with a small OpenAI model;
- Garak scanner-style coverage expansion is enabled in static-pattern mode.

The only intentionally non-live setting is `GARAK_TARGET_MODEL = None`. That means Garak contributes scanner-style coverage patterns to the dataset, but it does not run a heavy live scan against a target model from this dataset notebook. Live target evaluation belongs in notebook 02.

If you want a no-cost deterministic run, set `ENABLE_LLM_GENERATOR=False`, `ENABLE_DEEPTEAM=False`, and keep `DEEPTEAM_GENERATION_MODE="local_template"`.


In [ ]:
from dataclasses import replace

from finance_redteam.config import DEFAULT_CONFIG_PATH, load_benchmark_config
from finance_redteam.deepteam_adapter import DeepTeamExpansionConfig
from finance_redteam.garak_adapter import GarakExpansionConfig
from finance_redteam.llm_generator import LLMGeneratorConfig

CONFIG_PATH = Path("configs/finance_benchmark.yaml")
config = load_benchmark_config(CONFIG_PATH)

# ---------------------------------------------------------------------------
# Production demonstration profile
# ---------------------------------------------------------------------------
# Goal: exercise all available generation paths without creating an unbounded
# quota/cost event inside a notebook. Increase limits only after reviewing the
# output quality and coverage report.

PROFILE_NAME = "production_balanced_full_feature"

# Domain and dataset size.
# 5-10 prompts per category keeps the base benchmark broad enough for governance
# review while still easy to inspect in a notebook.
DOMAIN_PACK = "banking_finance"  # or a generated domain-pack YAML path
MIN_PER_CATEGORY = 5
MAX_PER_CATEGORY = 10

# Feature toggles.
# These are all enabled for the production demonstration profile.
ENABLE_SEED_SOURCES = True
ENABLE_GARAK_CORPUS = True
ENABLE_LOCAL_MUTATION_ORCHESTRATOR = True
ENABLE_LLM_GENERATOR = True
ENABLE_DEEPTEAM = True
ENABLE_GARAK_SCANNER_EXPANSION = True

# Seed-source controls.
# These adapters bring in framework-inspired starter ideas. They are useful for
# bootstrapping a new domain and for making sure the dataset is not only manually
# authored.
SEED_SOURCE_NAMES = ["owasp", "mitre_atlas", "garak"]
SEED_SOURCE_MAX_ITEMS = 60

# Local orchestration controls.
# These deterministic mutations are stable and audit-friendly. They should stay
# enabled even when LLM generation is used, because they provide reproducible
# coverage and lineage.
ORCHESTRATION_MAX_RECORDS = 30
ORCHESTRATION_VARIANTS_PER_RECORD = 3
ORCHESTRATION_INCLUDE_MULTI_TURN = True
ORCHESTRATION_MAX_MULTI_TURN_PLANS = 8

# LLM generation controls.
# gpt-4.1-nano is chosen as the default small model for low-latency, low-cost
# generation. The cap of 12 records is enough to prove the feature path and add
# variety without overwhelming manual review.
LLM_PROVIDER = "openai"       # openai or gemini
LLM_MODEL = "gpt-4.1-nano"
LLM_API_KEY_ENV = "OPENAI_API_KEY"
LLM_MAX_SEED_RECORDS = 6
LLM_VARIANTS_PER_SEED = 2
LLM_MAX_RECORDS = 12

# DeepTeam controls.
# generation_mode="llm" verifies real DeepTeam simulator-backed generation.
# The vulnerability and attack-type lists focus on high-value LLM app risks for
# finance: prompt injection, privacy leakage, RAG exfiltration, tool misuse, and
# unsafe advice. The max record cap keeps this notebook suitable for repeated
# production demos.
DEEPTEAM_GENERATION_MODE = "llm"  # local_template or llm
DEEPTEAM_SIMULATOR_PROVIDER = "openai"
DEEPTEAM_SIMULATOR_MODEL = "gpt-4.1-nano"
DEEPTEAM_API_KEY_ENV = "OPENAI_API_KEY"
DEEPTEAM_VULNERABILITIES = [
    "prompt_injection",
    "pii_leakage",
    "credential_leakage",
    "rag_exfiltration",
    "unsafe_financial_advice",
    "tool_misuse",
]
DEEPTEAM_ATTACK_TYPES = [
    "jailbreak",
    "developer_override",
    "system_prompt_extraction",
    "rag_exfiltration",
    "data_extraction",
    "tool_misuse",
]
DEEPTEAM_MAX_SEED_RECORDS = 8
DEEPTEAM_VARIANTS_PER_SEED = 1
DEEPTEAM_ATTACKS_PER_VULNERABILITY_TYPE = 1
DEEPTEAM_MAX_LLM_RECORDS = 8

# Garak corpus seed-mining controls.
# This mines Garak's installed probe/prompt material, reduces it, and adapts it
# into banking/finance-safe seed prompts. The probe allowlist is intentionally
# focused on instruction attacks, leakage, obfuscation, and misinformation.
GARAK_CORPUS_PROBES = [
    "promptinject",
    "dan",
    "encoding",
    "sysprompt_extraction",
    "leakreplay",
    "misleading",
    "web_injection",
    "latentinjection",
    "packagehallucination",
]
GARAK_CORPUS_MAX_RAW = 1000
GARAK_CORPUS_MAX_TOTAL_SEEDS = 25
GARAK_CORPUS_MAX_PER_PROBE = 5
GARAK_CORPUS_DOMAIN_TERMS = [
    "banking",
    "finance",
    "customer",
    "account",
    "payment",
    "compliance",
    "document",
    "assistant",
]

# Garak scanner-style expansion controls.
# Enabled for coverage expansion. GARAK_TARGET_MODEL stays None here so this
# notebook does not run a live Garak target scan. Use notebook 02 for live target
# evaluation.
GARAK_PROBE_FAMILIES = [
    "prompt_injection",
    "data_leakage",
    "jailbreak",
    "encoding_obfuscation",
    "rag_exfiltration",
    "tool_misuse",
    "hallucination",
    "misinformation",
    "model_extraction",
]
GARAK_MAX_PATTERNS = 12
GARAK_TARGET_MODEL = None

config = config.model_copy(update={
    "domain_pack": DOMAIN_PACK,
    "min_per_category": MIN_PER_CATEGORY,
    "max_per_category": MAX_PER_CATEGORY,
    "use_orchestrator": ENABLE_LOCAL_MUTATION_ORCHESTRATOR,
    "use_llm_generator": ENABLE_LLM_GENERATOR,
    "use_deepteam": ENABLE_DEEPTEAM,
    "use_garak": ENABLE_GARAK_SCANNER_EXPANSION,
    "seed_sources": config.seed_sources.model_copy(update={
        "enabled": ENABLE_SEED_SOURCES,
        "sources": SEED_SOURCE_NAMES,
        "max_items": SEED_SOURCE_MAX_ITEMS,
    }),
    "garak_corpus": config.garak_corpus.model_copy(update={
        "enabled": ENABLE_GARAK_CORPUS,
        "probe_allowlist": GARAK_CORPUS_PROBES,
        "max_raw_candidates": GARAK_CORPUS_MAX_RAW,
        "max_total_seeds": GARAK_CORPUS_MAX_TOTAL_SEEDS,
        "max_per_probe": GARAK_CORPUS_MAX_PER_PROBE,
        "domain_terms": GARAK_CORPUS_DOMAIN_TERMS,
    }),
    "orchestration": replace(config.orchestration,
        enabled=ENABLE_LOCAL_MUTATION_ORCHESTRATOR,
        max_records=ORCHESTRATION_MAX_RECORDS,
        variants_per_record=ORCHESTRATION_VARIANTS_PER_RECORD,
        include_multi_turn_plans=ORCHESTRATION_INCLUDE_MULTI_TURN,
        max_multi_turn_plans=ORCHESTRATION_MAX_MULTI_TURN_PLANS,
    ),
    "llm_generator": replace(config.llm_generator,
        provider=LLM_PROVIDER,
        model=LLM_MODEL,
        api_key_env=LLM_API_KEY_ENV,
        max_seed_records=LLM_MAX_SEED_RECORDS,
        variants_per_seed=LLM_VARIANTS_PER_SEED,
        max_records=LLM_MAX_RECORDS,
    ),
    "deepteam": replace(config.deepteam,
        generation_mode=DEEPTEAM_GENERATION_MODE,
        simulator_provider=DEEPTEAM_SIMULATOR_PROVIDER,
        simulator_model=DEEPTEAM_SIMULATOR_MODEL,
        api_key_env=DEEPTEAM_API_KEY_ENV,
        vulnerabilities=DEEPTEAM_VULNERABILITIES,
        attack_types=DEEPTEAM_ATTACK_TYPES,
        max_seed_records=DEEPTEAM_MAX_SEED_RECORDS,
        variants_per_seed=DEEPTEAM_VARIANTS_PER_SEED,
        attacks_per_vulnerability_type=DEEPTEAM_ATTACKS_PER_VULNERABILITY_TYPE,
        max_llm_records=DEEPTEAM_MAX_LLM_RECORDS,
    ),
    "garak": config.garak.model_copy(update={
        "enabled": ENABLE_GARAK_SCANNER_EXPANSION,
        "target_model": GARAK_TARGET_MODEL,
        "probe_families": GARAK_PROBE_FAMILIES,
        "max_patterns": GARAK_MAX_PATTERNS,
        "include_static_patterns": True,
        "include_parsed_findings": True,
    }),
})

parameter_justification = {
    "profile": PROFILE_NAME,
    "base_dataset": "5-10 prompts per category gives broad risk coverage without making notebook review difficult.",
    "seed_sources": "OWASP, MITRE ATLAS, and Garak-style source adapters reduce manual cold-start work and improve framework coverage.",
    "garak_corpus": "Mines a bounded subset of Garak's installed prompt/probe corpus, then reduces and adapts it to finance instead of blindly importing everything.",
    "local_orchestration": "Deterministic mutations and multi-turn plans provide reproducible, lineage-aware variants for audit and regression testing.",
    "llm_generator": "OpenAI gpt-4.1-nano adds creative domain-specific mutations with a small cost/latency footprint; capped at 12 records for reviewability.",
    "deepteam": "Live DeepTeam simulator-backed generation is enabled for adversarial variation quality, capped at 8 records to avoid runaway API use.",
    "garak_scanner": "Static scanner-style Garak coverage is enabled; live target scan is kept out of this notebook and belongs in evaluation notebook 02.",
}

print("Active profile:", PROFILE_NAME)
print("Feature toggles:")
print(json.dumps({
    "seed_sources": ENABLE_SEED_SOURCES,
    "garak_corpus": ENABLE_GARAK_CORPUS,
    "local_orchestrator": ENABLE_LOCAL_MUTATION_ORCHESTRATOR,
    "llm_generator": ENABLE_LLM_GENERATOR,
    "deepteam": ENABLE_DEEPTEAM,
    "garak_scanner_expansion": ENABLE_GARAK_SCANNER_EXPANSION,
}, indent=2))
print("\nParameter justification:")
print(json.dumps(parameter_justification, indent=2))
print("\nResolved config preview:")
print(yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False)[:5000])


## 4. Inspect Domain Pack And Seed Guidance

Run this before building. It tells you what domain is being tested, which personas/contexts are available, and how seed prompts should be written.

For a new domain, this is the section you show stakeholders to explain how the benchmark is anchored in domain risk.


In [ ]:
from finance_redteam.domain_pack import get_domain_pack
from finance_redteam.seed_prompts import build_seed_authoring_starter
from finance_redteam.taxonomy import build_default_taxonomy

pack = get_domain_pack(config.domain_pack)
categories = build_default_taxonomy(pack)
starter = build_seed_authoring_starter(pack)

print("Domain pack:", pack.domain_id, "-", pack.display_name)
print("Description:", pack.description)
print("Personas:", ", ".join(pack.personas[:10]))
print("Contexts:", ", ".join(pack.contexts[:10]))
print("Risk categories:", len(categories))
print("First 5 risks:")
for category in categories[:5]:
    print(f"- {category.category_id}: {category.name}")

print("\nSeed starter purpose:")
print(starter.get("purpose", "No purpose text available."))

print("\nSeed authoring questions:")
for question in starter.get("questions", []):
    print(f"- {question}")

print("\nAuthoring rules:")
for rule in starter.get("authoring_rules", []):
    print(f"- {rule}")

print("\nRecommended seed shape:")
print(json.dumps(starter.get("recommended_seed_shape", {}), indent=2))


## 5. Preview Available Expansion Selectors

Use these values in the toggle cell above. This prevents guessing names for DeepTeam vulnerabilities, DeepTeam attack types, Garak probe families, or mutation strategies.


In [ ]:
from finance_redteam.deepteam_adapter import (
    is_deepteam_available,
    supported_deepteam_attack_types,
    supported_deepteam_vulnerabilities,
)
from finance_redteam.garak_adapter import (
    is_garak_available,
    supported_garak_attack_types,
    supported_garak_probe_families,
)
from finance_redteam.mutation_strategies import DEFAULT_MUTATION_STRATEGIES

print("DeepTeam installed:", is_deepteam_available())
print("DeepTeam vulnerabilities:", supported_deepteam_vulnerabilities())
print("DeepTeam attack types:", supported_deepteam_attack_types())
print()
print("Garak installed:", is_garak_available())
print("Garak probe families:", supported_garak_probe_families())
print("Garak attack types:", supported_garak_attack_types())
print()
print("Local mutation strategies:", [strategy.name for strategy in DEFAULT_MUTATION_STRATEGIES])


## 6. Optional Preview: Garak Corpus As Seed Material

This does not evaluate a model. It only looks inside the installed Garak package, extracts candidate probe/prompt strings, reduces the size, and adapts selected items into domain seed prompts.

Use this when you want Garak's built-in adversarial corpus to help start or enrich your domain-specific seeds.


In [ ]:
from finance_redteam.garak_corpus import (
    extract_garak_corpus_candidates,
    garak_candidates_to_seed_prompts,
    reduce_garak_corpus_candidates,
)

PREVIEW_GARAK_CORPUS = True

if PREVIEW_GARAK_CORPUS and config.garak_corpus.enabled:
    corpus_cfg = config.garak_corpus.corpus_config()
    raw_candidates = extract_garak_corpus_candidates(pack, corpus_cfg)
    reduced_candidates = reduce_garak_corpus_candidates(raw_candidates, corpus_cfg)
    preview_seeds = garak_candidates_to_seed_prompts(reduced_candidates, categories, pack)
    print("Raw Garak candidates:", len(raw_candidates))
    print("Reduced candidates:", len(reduced_candidates))
    print("Domain-adapted seed prompts:", len(preview_seeds))
    for seed in preview_seeds[:5]:
        print("-", seed.category_id, "|", seed.attack_type, "|", seed.prompt[:220])
else:
    print("Garak corpus preview skipped. Set PREVIEW_GARAK_CORPUS=True and ENABLE_GARAK_CORPUS=True.")


## 7. Optional Smoke Test: DeepTeam And Garak Expansion

This is a tiny integration check for dependency changes. It confirms that DeepTeam-style expansion and Garak scanner-style expansion can produce valid records before you run the full build.

For the production profile, this is **off by default** because the full build already exercises the enabled generation paths. Turn it on only when you changed DeepTeam, Garak, API keys, or selector values and want a quick preflight check.


In [ ]:
from finance_redteam.deepteam_adapter import generate_deepteam_variants
from finance_redteam.garak_adapter import run_garak_scan
from finance_redteam.local_generator import seeds_to_records
from finance_redteam.seed_prompts import build_seed_prompts
from finance_redteam.validator import validate_records

RUN_EXPANSION_SMOKE_TEST = False

if RUN_EXPANSION_SMOKE_TEST:
    seed_prompts = build_seed_prompts(categories, per_category=1, domain_pack=pack)[:3]
    seed_records = seeds_to_records(seed_prompts, categories, source="seed", domain_pack=pack)

    smoke_deepteam = generate_deepteam_variants(seed_records, expansion_config=config.deepteam)
    smoke_garak = run_garak_scan(target_model=None, expansion_config=config.garak.expansion_config())

    smoke_records = smoke_deepteam[:5] + smoke_garak[:5]
    smoke_validation = validate_records(smoke_records)

    print("Seed records used:", len(seed_records))
    print("DeepTeam smoke records:", len(smoke_deepteam))
    print("Garak scanner smoke records:", len(smoke_garak))
    print("Smoke validation valid:", smoke_validation.valid)
    print("Smoke validation errors:", len(smoke_validation.errors))
    for record in smoke_records[:8]:
        print(f"- {record.attack_id} | source={record.source} | attack_type={record.attack_type} | category={record.risk_category}")
else:
    print("Expansion smoke test skipped.")


## 8. Build And Export Dataset

This runs the full configured pipeline:

1. Build/load taxonomy from the domain pack.
2. Generate baseline seed prompts.
3. Add seed-source items.
4. Optionally add Garak corpus seeds.
5. Apply local mutation orchestration and multi-turn plans.
6. Optionally run LLM generation.
7. Optionally run DeepTeam expansion.
8. Optionally run Garak scanner expansion.
9. Normalize, deduplicate, validate, attach lineage/coverage, and export JSONL + Promptfoo YAML.


In [ ]:
from finance_redteam.builder import build_records_from_config, export_build_result

RUN_FULL_BUILD = True

if RUN_FULL_BUILD:
    build_result = build_records_from_config(config)
    export_build_result(build_result, config)

    print("Total exported records:", len(build_result.records))
    print("Mutation records:", len(build_result.mutation_records))
    print("LLM records:", len(build_result.llm_records))
    print("DeepTeam records:", len(build_result.deepteam_records))
    print("Garak records:", len(build_result.garak_records))
    print("Validation valid:", build_result.validation.valid)
    print("Validation errors:", len(build_result.validation.errors))
    print("Review warnings:", len(build_result.validation.review_warnings))
    print("JSONL:", config.output.jsonl)
    print("Promptfoo YAML:", config.output.promptfoo_yaml)
else:
    print("Full build skipped. Set RUN_FULL_BUILD=True when ready.")


## 9. Coverage, Lineage, And Source Summary

This tells you what the benchmark contains and where records came from. Use it to explain coverage to stakeholders.


In [ ]:
if "build_result" not in globals():
    raise RuntimeError("Run the build cell first.")

records = build_result.records
print("Records by source:")
print(json.dumps(Counter(record.source for record in records), indent=2))
print("\nRecords by attack type, top 12:")
print(json.dumps(dict(Counter(record.attack_type for record in records).most_common(12)), indent=2))
print("\nRecords by risk category, top 12:")
print(json.dumps(dict(Counter(record.risk_category for record in records).most_common(12)), indent=2))
print("\nCoverage report:")
print(json.dumps(build_result.coverage.to_dict(), indent=2)[:4000])


## 10. Preview Exported Dataset

Each JSONL row is one red-team test case. The `prompt` or `attack_query` is what gets sent to the target model during evaluation. The expected behavior tells the evaluator what safe behavior looks like.


In [ ]:
from finance_redteam.exporters import load_jsonl

exported_records = load_jsonl(config.output.jsonl)
print("Exported records:", len(exported_records))
for record in exported_records[:3]:
    print("\n---")
    print("attack_id:", record.attack_id)
    print("source:", record.source)
    print("risk_category:", record.risk_category)
    print("attack_type:", record.attack_type)
    print("difficulty:", record.difficulty)
    print("lineage:", " > ".join(record.lineage))
    print("prompt:", (record.attack_query or record.prompt)[:500])
    print("expected_behavior:", record.expected_behavior[:300])


## 11. Next Step

Open `02_evaluate_attacks_gemini.ipynb` to evaluate the generated attacks against a target model.

Common next commands outside the notebook:

```bash
python -m finance_redteam.cli validate
npm run eval:openai:smoke
npm run eval:openai
```

For a new domain, repeat the same notebook after changing `DOMAIN_PACK` or `CONFIG_PATH`.
